<a href="https://colab.research.google.com/github/ravikeerthi7606/HousePricePrediction/blob/main/Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q mlflow joblib

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 661.9 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 47.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 97.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 65.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.7/268.7 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.7/148.7 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
#mlflow directory
import os
from pathlib import Path

# Opt-in to filesystem tracking backend
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

mlflow_dir = Path(
    "/content/drive/MyDrive/ML project/HousePricePrediction/mlruns"
)

mlflow_dir.mkdir(parents=True, exist_ok=True)

#Create an experiment
import mlflow

mlflow.set_tracking_uri(mlflow_dir.as_uri())

mlflow.set_experiment("House Price Prediction")

<Experiment: artifact_location='file:///content/drive/MyDrive/ML%20project/HousePricePrediction/mlruns/769246350787289816', creation_time=1791190156868, effective_trace_archival_retention=None, experiment_id='769246350787289816', last_update_time=1791190156868, lifecycle_stage='active', name='House Price Prediction', tags={}, trace_location=None, workspace='default'>

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shashanknecrothapa/ames-housing-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'ames-housing-dataset' dataset.
Path to dataset files: /kaggle/input/ames-housing-dataset


In [5]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

#DON'T FORGET TO CHANGE THE PATH
df = pd.read_csv('/kaggle/input/ames-housing-dataset/AmesHousing.csv')
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [38]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
import mlflow
import mlflow.sklearn
import joblib
import warnings
warnings.filterwarnings('ignore')

In [7]:
X = df.drop(columns='SalePrice')
y = df['SalePrice']

numeric_columns = X.select_dtypes(include=np.number).columns
categorical_columns = X.select_dtypes(exclude=np.number).columns


print(f'numerical columns: {len(numeric_columns)}')
print(f'categorical columns: {len(categorical_columns)}')

numerical columns: 38
categorical columns: 43


In [15]:
#imputation
numeric_pipeline = Pipeline(
    steps=[('impute', SimpleImputer(strategy='median'))]
          #  ('scale', StandardScaler())]
)
categorical_pipeline = Pipeline(
    steps=[('impute', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore'))]
  )

#column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_pipeline, numeric_columns),
        ('categorical', categorical_pipeline, categorical_columns)
    ])

In [9]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42)

X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=42)

In [39]:
#Linear regression model
# linear_model = LassoCV(cv=5,random_state=69)

#Decision Tree
dt = DecisionTreeRegressor(random_state=69)

#Gradient Boosting
gb = GradientBoostingRegressor(random_state=69)

# #Random Forest
# rf = RandomForestRegressor(random_state=69)

# #Ada Boost
# adaboost = AdaBoostRegressor(random_state=69)

#Pipeline
linear_pipeline = Pipeline(
    steps=[('preprocessor', preprocessor),
    ('Trees',gb)]
)

In [40]:
#mlflow
from pathlib import Path

with mlflow.start_run(run_name='Gradient Boosting Regressor'):
  linear_pipeline.fit(X_train, y_train)

  rmse = np.sqrt(mean_squared_error(y_test, linear_pipeline.predict(X_test)))
  mae = mean_absolute_error(y_test, linear_pipeline.predict(X_test))
  r2 = r2_score(y_test, linear_pipeline.predict(X_test))

  # Ensure target directory exists
  model_path = Path("/content/drive/MyDrive/ML project/HousePricePrediction/models/GradientBoosting.joblib")
  model_path.parent.mkdir(parents=True, exist_ok=True)

  joblib.dump(linear_pipeline, model_path)

  mlflow.sklearn.log_model(
    linear_pipeline,
    "Gradient boost Regressor",
    serialization_format='cloudpickle'
  )
  # mlflow.log_param('features importance', linear_pipeline.feature_importance_)
  mlflow.autolog()
  mlflow.log_metric('mse', rmse)
  mlflow.log_metric('mae', mae)
  mlflow.log_metric('r2', r2)

2026/10/05 19:05:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/05 19:05:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/10/05 19:05:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/10/05 19:05:37 INFO mlflow.tracking.fluent: Autologging successfully enabled for googl

In [41]:
#MAE
print(mean_absolute_error(y_test, linear_pipeline.predict(X_test)))
#mse
print(np.sqrt(mean_squared_error(y_test, linear_pipeline.predict(X_test))))
#r2
print(r2_score(y_test, linear_pipeline.predict(X_test)))

14683.75062351251
25288.19219803996
0.911283124013595


In [42]:
experiment = mlflow.get_experiment_by_name(
    "House Price Prediction"
)

runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)

runs[['tags.mlflow.runName','metrics.mse','metrics.mae','metrics.r2']]

,tags.mlflow.runName,metrics.mse,metrics.mae,metrics.r2
0,Gradient Boosting Regressor,25288.192198,14683.750624,0.911283
1,Adaboost Regressor,34274.848050,25109.502708,0.837025
2,Random Tree Regressor,31689.271192,19655.193293,0.860686
3,Random Tree Regressor,NaN,NaN,NaN
4,Decision Tree Regressorr,36133.077528,24414.740909,0.818874
5,LassoCV,30455.034315,16043.502104,0.871327
6,Ridge,31238.351452,16971.016574,0.864622
7,Linear Regression,32492.015632,16891.331812,0.853538
8,Linear Regression,NaN,NaN,NaN
9,Linear Regression,NaN,NaN,NaN


In [43]:
runs

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.r2,metrics.mae,metrics.training_mean_squared_error,metrics.r2_score-2_X_test,...,params.Trees__bootstrap,params.Trees__oob_score,params.Trees__monotonic_cst,params.Trees__n_jobs,tags.mlflow.source.name,tags.estimator_class,tags.mlflow.user,tags.mlflow.runName,tags.estimator_name,tags.mlflow.source.type
0,3c00cce4184f4eb9ba92f2d3db71cd16,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 19:05:23.083000+00:00,2026-10-05 19:05:37.654000+00:00,0.911283,14683.750624,2.270672e+08,0.911283,...,None,None,None,None,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,sklearn.pipeline.Pipeline,root,Gradient Boosting Regressor,Pipeline,NOTEBOOK
1,1f8ba719bb174071b518b4de6e1592cd,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 18:58:00.523000+00:00,2026-10-05 18:58:14.599000+00:00,0.837025,25109.502708,8.747149e+08,0.837025,...,None,None,None,None,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,sklearn.pipeline.Pipeline,root,Adaboost Regressor,Pipeline,NOTEBOOK
2,7be9bfca4f5948d19ac9a13c26ccabc1,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 18:54:28.227000+00:00,2026-10-05 18:54:44.790000+00:00,0.860686,19655.193293,5.942585e+08,0.860686,...,True,False,None,None,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,sklearn.pipeline.Pipeline,root,Random Tree Regressor,Pipeline,NOTEBOOK
3,0b7476924fcd494582b2626439e1b99a,769246350787289816,FAILED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 18:53:41.098000+00:00,2026-10-05 18:53:55.464000+00:00,NaN,NaN,5.942585e+08,NaN,...,True,False,None,None,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,sklearn.pipeline.Pipeline,root,Random Tree Regressor,Pipeline,NOTEBOOK
4,880c210ba8b24152af0d0c950eaa1a37,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 18:44:05.664000+00:00,2026-10-05 18:44:14.728000+00:00,0.818874,24414.740909,NaN,NaN,...,None,None,None,None,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,root,Decision Tree Regressorr,None,NOTEBOOK
5,31e77ae06a4642caa332b02643be3615,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 18:36:11.452000+00:00,2026-10-05 18:36:30.319000+00:00,0.871327,16043.502104,NaN,NaN,...,None,None,None,None,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,root,LassoCV,None,NOTEBOOK
6,40e9a620caab43cba581756300c29f2c,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 09:06:56.801000+00:00,2026-10-05 09:07:02.833000+00:00,0.864622,16971.016574,NaN,NaN,...,None,None,None,None,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,root,Ridge,None,NOTEBOOK
7,f1b38b33b73048108945549c23cadc4d,769246350787289816,FINISHED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 08:59:35.693000+00:00,2026-10-05 08:59:43.164000+00:00,0.853538,16891.331812,NaN,NaN,...,None,None,None,None,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,root,Linear Regression,None,NOTEBOOK
8,34960b86d2524cb9b578f539b26012cf,769246350787289816,FAILED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 08:59:18.545000+00:00,2026-10-05 08:59:19.701000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,None,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,root,Linear Regression,None,NOTEBOOK
9,92bd3842931b45a9a95fc2768580c3a3,769246350787289816,FAILED,file:///content/drive/MyDrive/ML%20project/Hou...,2026-10-05 08:58:30.262000+00:00,2026-10-05 08:58:30.933000+00:00,NaN,NaN,NaN,NaN,...,None,None,None,None,fileId=14kcRMAGna1d66txEO4_RjRtLeS5ySY86,None,root,Linear Regression,None,NOTEBOOK
